# Getting data in and out

This is again just the prologue to allow the notebooks running standalone in a local Jupyter installation and in Google Colab.

In [ ]:
import os
try:
    import google.colab
    IN_COLAB = True
    if not os.path.isdir("/content/oreilly-duckdb"):
        os.system("git clone https://github.com/datanizing/oreilly-duckdb")
    data = "/content/oreilly-duckdb/data"
except:
    IN_COLAB = False
    data = "data"

In [ ]:
import duckdb

## Query CSV like a table

You have already seen this in the last notebook. `duckdb` allows
querying CSV files like they were full-fledged database tables.

In [ ]:
duckdb.sql(f"SELECT Location, AVG(COLUMNS('.*Temp.*')) FROM '{data}/weatherAUS.csv.zst' \
                    GROUP BY ALL ORDER BY Location").pl()

Sometimes, the `SUMMARIZE` function might be especially helpful:

In [ ]:
duckdb.sql(f"SUMMARIZE FROM '{data}/weatherAUS.csv.zst'").pl()

## Query Excel files

What works nicely for CSV files can also be applied to Excel files.
`duckdb` can read `.xlsx` files and also work with different sheets.
`duckdb` tries to infer the data types according to the cell format
which has been set in Excel:

In [ ]:
duckdb.sql(f"FROM '{data}/example.xlsx'").pl()

If you are *just* interested in the data types or the structure of
the data, you could use `DESCRIBE`:

In [ ]:
duckdb.sql(f"DESCRIBE FROM '{data}/example.xlsx'").pl()

The `SUMMARIZE` function is also very useful for Excel files:

In [ ]:
duckdb.sql(f"SUMMARIZE FROM '{data}/example.xlsx'").pl()

## Query XML files

Natively, `duckdb` cannot read `XML` files. However, there is a
*community extensions* which adds this capability called `webbed`.

`XML` files are not anymore very common in our days and have 
been overtaken by `JSON` files. However, occassionally you will
still encounter these file types. One example is the so-called
`sitemap.xml` which lists files on websites. O'Reilly luckily
provides these files. For these course, we are interested in
the `video` section which I have downloaded and added to the
repository. You can analyze the files via `duckdb`:

In [ ]:
# install the community extension, you only need to perform this once
duckdb.sql("INSTALL webbed FROM community")

In [ ]:
# whenever you need the extension, you can load it now
duckdb.sql("LOAD webbed")

In [ ]:
# SELECT works directly on the XML file(s)
duckdb.sql(f"SELECT * FROM '{data}/video-sitemap/video-chapters-p1.xml'").pl()

As you can see, the column `video` contains structured information.
That's due to the structure of the `XML` file which `duckdb` tries
to reproduce. Take a look at the `XML` file directly:

In [ ]:
import xml.etree.ElementTree as ET

element = ET.XML(open(f"{data}/video-sitemap/video-chapters-p1.xml").read())
ET.indent(element)
print(ET.tostring(element, encoding='unicode'))

How can we *unpack* the structure in `duckdb`? Turns out this
is quite simple:

In [ ]:
duckdb.sql(f"SELECT * EXCLUDE video, unnest(video) FROM '{data}/video-sitemap/video-chapters-p1.xml'").pl()

Unfortunately, `webbed` cannot work with compressed files. All the
video sitemap data is too large for the repository (O'Reilly is
very productive), so I have converted the `XML` files to `JSON` with
`xq`, compressed them and we will take a look at them in the next
section!

## Query JSON files

`duckdb` can work with `JSON` files natively, which is a big plus as the
format becomes so ubiquitous. The `JSON` reader is quite flexible as it 
also understands the `JSONL` file where each line of the file contains a
separate `JSON` object. Let's read the first sitemap file:

In [ ]:
duckdb.sql(f"SELECT * FROM '{data}/video-sitemap/video-chapters-p1.json'").pl()

Even better, the reader can also handle *compressed* files. Let's read the 
file again in the compressed version and unpack the `video:video` column
at the same time:

In [ ]:
duckdb.sql(f"""SELECT * EXCLUDE "video:video", unnest("video:video") 
               FROM '{data}/video-sitemap/video-chapters-p1.json.zst'""").pl()

The `video:` prefix originates from the conversion of the nested `XML` file.
We can work with these prefixes, but we have to use quotes all the time which
is not very convenient. `duckdb` has very powerful, regex-based strategies for
converting column names. We will use that to remove the `video:` prefix now:

In [ ]:
duckdb.sql(f"""WITH videos AS (SELECT * EXCLUDE "video:video", unnest("video:video")
                               FROM '{data}/video-sitemap/video-chapters-p*.json.zst')
               SELECT COLUMNS('video:(\\w*)') AS '\\1' FROM videos""").pl()

This looks much better! However, there is not just one `JSON` file,
but several hundred. Here comes another *superpower* of `duckdb`
which allows wildcard in the filenames and makes it extremely easy
to select all videos. 

The `filename` columns is just here for illustrative purposes, in
this case we don't need it but it can be very handy:

In [ ]:
duckdb.sql(f"""SELECT filename, * EXCLUDE "video:video", unnest("video:video") 
               FROM '{data}/video-sitemap/video-chapters-p*.json.zst'""").pl()

Even though this selection process covers *several hundred* files, it
is still quite fast. However, most of the time you want to save these
aggregated data in a single file.

## Convert to `parquet` format

Often, saving data in `parquet` format is an excellent choice. `parquet`
is a column-oriented data format and highly optimized for selecting
indivudal columns or aggregate them. Moreover, it is platform-independent
and you can read the files also directly in `pandas` or `polars`.

Creating a `parquet` file is very easy in `duckdb` as the `COPY` statement
can be used directly on the last SQL statement from above.

In [ ]:
duckdb.sql(f"""COPY (WITH videos AS (SELECT * EXCLUDE "video:video", unnest("video:video")
                                    FROM '{data}/video-sitemap/video-chapters-p*.json.zst')
                     SELECT COLUMNS('video:(\\w*)') AS '\\1' FROM videos)
               TO 'video-sitemap.parquet'""")

`duckdb` offers a bunch of parameters for saving in `parquet` format like the compression
factor etc. For this course, it is not (yet) important.

Selecting the data now looks exactly like you know it from CSV files:

In [ ]:
duckdb.sql("SELECT * FROM 'video-sitemap.parquet'").pl()

The statements are much simpler now, we'd like to find out the oldest
videos from O'Reilly:

In [ ]:
duckdb.sql("SELECT * FROM 'video-sitemap.parquet' ORDER BY publication_date").pl()

Also, finding the longest running total videos (all chapters aggregated)
is very fast. Although the `parquet` file has roughly 500,000 rows, that's
a piece of cake for `duckdb`.

In [ ]:
duckdb.sql("""SELECT title, sum(duration::INTEGER) AS total_duration
              FROM 'video-sitemap.parquet' GROUP BY ALL ORDER BY total_duration DESC""").pl()

Some videos run quite long. Spark is arguably much more complicated
than `duckdb`, but are there also videos about `duckdb`?

In [ ]:
duckdb.sql("SELECT * FROM 'video-sitemap.parquet' WHERE LOWER(title) LIKE '%duckdb%' ORDER BY publication_date").pl()

If you need additional information, seeing one of these courses might be a good option.

Let's perform a final check and see if there are duplicate titles in the sitemap:

In [ ]:
duckdb.sql("""SELECT title, description, COUNT(*) AS count
              FROM 'video-sitemap.parquet' GROUP BY ALL HAVING count>1""").pl()

In [ ]:
duckdb.sql("SELECT * FROM 'video-sitemap.parquet' WHERE description LIKE 'Excited about augmented realit%'").pl().glimpse(return_type="frame")

Apart from the `duration`, everything is the same. Maybe we have some optimization ideas for O'Reilly 😆.

## Use `duckdb`'s native format

So far, we have used different file formats for querying data. However, `duckdb` also
has its own database format. If you use the CLI client, you can invoke `duckdb` with
a database file name and everything will be persisted there (instead of just living
in memory).

Let's first remove any database file which might exist from a prior run of the
notebook:

In [ ]:
import os
try:
    os.remove('video-sitemap.duckdb')
except:
    pass

In Python (and also in the CLI), we can `ATTACH` database files. This
database can then be used for persisting tables etc.

In [ ]:
duckdb.sql("ATTACH 'video-sitemap.duckdb'")

We now have two databases, on in memory and one on disk:

In [ ]:
duckdb.sql("SHOW DATABASES")

If we want to use the persistent database, we have to `USE` it first.
Otherwise, everything goes into the memory database:

In [ ]:
duckdb.sql("""USE "video-sitemap" """)

Create a persistent table by selecting the whole `parquet` file:

In [ ]:
duckdb.sql("CREATE TABLE videos AS SELECT * FROM 'video-sitemap.parquet'")

Check wether that has worked:

In [ ]:
duckdb.sql("SELECT * FROM videos").pl()

Create a persistent view:

In [ ]:
duckdb.sql("CREATE VIEW videos2025 AS SELECT * FROM videos \
            WHERE date_part('year', publication_date)='2025'")

If we want to `DETACH` the database, we first have to `USE` the
memory database again, otherwise `duckdb` would not know where
to save tables:

In [ ]:
duckdb.sql("""USE memory""")

Now we can `DETACH` it:

In [ ]:
duckdb.sql("""DETACH "video-sitemap" """)

Observe the size of the database file compared to the `parquet` file:

In [ ]:
!ls -l video-sitemap*

If you try to `SELECT` from the table without attaching the database,
you will get an error.

In [ ]:
duckdb.sql("SELECT * FROM videos").pl() 

After attaching, everything will be there again - persistence worked!

In [ ]:
duckdb.sql("ATTACH 'video-sitemap.duckdb'")
duckdb.sql("""USE "video-sitemap" """)
duckdb.sql("SELECT * FROM videos").pl() 

In [ ]:
duckdb.sql("SELECT * FROM videos2025").pl() 